# DF-MVR: 2D 얼굴 사진 → 3D 모델

**순서:**
1. 환경 세팅
2. 모델 파일 다운로드 (Google Drive)
3. 얼굴 사진 3장 업로드
4. 실행 → `.obj` 파일 저장

> ⚠️ 상단 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 시작하세요!

## Step 1. 환경 세팅

In [ ]:
# GPU 확인
!nvidia-smi

In [ ]:
# PyTorch 버전 확인 (Colab 기본 버전 사용)
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 레포 클론
!git clone https://github.com/weiguangzhao/DF_MVR.git
%cd DF_MVR

In [ ]:
# 패키지 설치
!pip install -q matplotlib tensorboardX pyyaml h5py scipy mtcnn face-alignment
!pip install -q mmengine

In [ ]:
# pytorch3d 설치 (Colab T4용)
import torch
pyt_version_str = torch.__version__.replace("+", "%2B")
version_str = "py3{}_cu{}_pyt{}".format(
    "".join(str(sys.version_info[1])),
    torch.version.cuda.replace(".",""),
    pyt_version_str.replace(".","")
)
import sys
pyt_version_str = torch.__version__.replace("+", "%2B")
cuda_ver = torch.version.cuda.replace(".","")
py_ver = f"{sys.version_info.major}{sys.version_info.minor}"
pyt_ver = torch.__version__.split("+")[0].replace(".","")
version_str = f"py3{py_ver}_cu{cuda_ver}_pyt{pyt_ver}"
print('Installing pytorch3d for:', version_str)
!pip install -q --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/{version_str}/download.html

## Step 2. 모델 파일 다운로드

In [ ]:
# pretrain 폴더 생성
!mkdir -p pretrain
!mkdir -p lib/BFM/model_basis/BFM2009

In [ ]:
# gdown으로 Google Drive 파일 다운로드
!pip install -q gdown

# face_mask.pth
!gdown '1wldpAhrPUVYy7JMtgt6aaYkMID_VsalI' -O pretrain/face_mask.pth

# DF_MVR pretrain (000000279.pth)
!gdown '1rA8V6-ZP8wLp531NuM2TF8PgbqdsaiT8' -O pretrain/000000279.pth

# BFM_model_front.mat
!gdown '1XJZVxpJLWQz5jSNJiFTjI6ZRkowMTGAp' -O lib/BFM/model_basis/BFM2009/BFM_model_front.mat

print('다운로드 완료!')
!ls -lh pretrain/
!ls -lh lib/BFM/model_basis/BFM2009/

## Step 3. 내 얼굴 사진 업로드

**사진 3장 필요:**
- `my_face0.png` — 왼쪽 45도
- `my_face1.png` — 정면
- `my_face2.png` — 오른쪽 45도

> 💡 팁: 스마트폰으로 같은 자리에서 고개만 돌려서 3장 찍으면 됨. 배경 단순할수록 좋음.
> 이미지 크기는 자동으로 224x224로 리사이즈됩니다.

In [ ]:
from google.colab import files
from PIL import Image
import os

# 업로드
print('사진 3장을 순서대로 업로드하세요: 왼쪽 → 정면 → 오른쪽')
uploaded = files.upload()

In [ ]:
# 업로드된 파일을 samples 폴더에 224x224로 저장
import os
from PIL import Image

os.makedirs('samples/0', exist_ok=True)
os.makedirs('samples/1', exist_ok=True)
os.makedirs('samples/2', exist_ok=True)

uploaded_files = list(uploaded.keys())
print('업로드된 파일:', uploaded_files)

if len(uploaded_files) != 3:
    print('⚠️ 파일 3개가 필요합니다! 다시 업로드하세요.')
else:
    for i, fname in enumerate(sorted(uploaded_files)):
        img = Image.open(fname).convert('RGB').resize((224, 224))
        save_path = f'samples/{i}/myface{i}.png'
        img.save(save_path)
        print(f'저장: {save_path}')
    print('✅ 준비 완료!')

In [ ]:
# 업로드된 사진 미리보기
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
titles = ['왼쪽 (0)', '정면 (1)', '오른쪽 (2)']
for i in range(3):
    img = Image.open(f'samples/{i}/myface{i}.png')
    axes[i].imshow(img)
    axes[i].set_title(titles[i])
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## Step 4. 3D 재구성 실행

In [ ]:
# test_sample.py의 파일명 규칙에 맞게 수정
# 규칙: samples/1/이름1.png 기준으로 samples/0/이름0.png, samples/2/이름2.png 탐색
# 이미 myface0, myface1, myface2로 저장했으므로 OK

!python test_sample.py --gpu 0

## Step 5. 결과 확인 & 다운로드

In [ ]:
# 결과 파일 확인
!ls -lh samples_results/obj/

In [ ]:
# .obj 파일 다운로드
from google.colab import files
import glob

obj_files = glob.glob('samples_results/obj/*.obj')
print('생성된 파일:', obj_files)
for f in obj_files:
    files.download(f)
    print(f'다운로드: {f}')

## Step 6. 3D 뷰어로 확인

다운받은 `.obj` 파일은 아래 사이트에서 바로 열 수 있어:
- **https://3dviewer.net** — 파일 드래그앤드롭으로 바로 확인
- **https://threejs.org/editor** — Three.js 기반 뷰어
- 또는 Blender에서 Import → Wavefront (.obj)

In [ ]:
# (선택) notebook 안에서 3D 미리보기
# plotly로 .obj 파싱해서 시각화
!pip install -q plotly

import plotly.graph_objects as go
import numpy as np

def load_obj(path):
    verts, faces = [], []
    with open(path) as f:
        for line in f:
            if line.startswith('v '):
                verts.append(list(map(float, line.split()[1:4])))
            elif line.startswith('f '):
                faces.append([int(x.split('/')[0])-1 for x in line.split()[1:4]])
    return np.array(verts), np.array(faces)

obj_files = glob.glob('samples_results/obj/*.obj')
if obj_files:
    verts, faces = load_obj(obj_files[0])
    fig = go.Figure(data=[
        go.Mesh3d(
            x=verts[:,0], y=verts[:,1], z=verts[:,2],
            i=faces[:,0], j=faces[:,1], k=faces[:,2],
            color='lightpink', opacity=1.0,
            lighting=dict(ambient=0.5, diffuse=0.8, specular=0.3),
        )
    ])
    fig.update_layout(title='내 얼굴 3D 모델', scene=dict(aspectmode='data'))
    fig.show()
else:
    print('결과 파일이 없습니다. Step 4를 먼저 실행하세요.')